# 第14回 演習：区分の表を、地図にする


---


## 前回からの接続


第9回・第10回は、数の列（顔の明るさ、車の性能）を主成分（principal component）の地図にしました。第11回・第13回は、数の列を使って、国をグループに分けました。

今日のデータには、数の列がありません。映画の「ジャンル」と「公開の年代」という、2つの区分を数えた表だけです。それでも、「どのジャンルと、どの年代が近いか」を1枚の地図で読めます。その方法が対応分析（correspondence analysis）です。今日は、これまでに学んだ書き方を総ざらいしながら進めます。


---


## 今日の分析目標


あなたは映画配信サービスの企画担当者です。「年代別の特集」を組むとき、どの年代にどのジャンルを合わせれば、その時代らしさが出るかを、1枚の地図と数で示したい。

ジャンルと年代に関係があるかを確かめ、対応分析で地図を描き、年代ごとに「多すぎる」ジャンルを数で確かめます。最後の「目標に答えられたか」で振り返りましょう。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- クロス表の「期待度数（expected frequency）」を計算し、カイ二乗で2つの区分の関係を確かめられる
- 対応分析で、行（ジャンル）と列（年代）を1枚の地図に置き、寄与率（proportion of variance explained）を読める
- 「実際 ÷ 期待」の倍率で、地図の読みを数で確かめられる
- これまでに学んだ書き方（配列の計算、`def`、`for`、辞書、f-string）を組み合わせて使える


---


## 0. 設定


### 最初に実行する：道具を呼び出す設定のセル

- 役割：このノートで使う道具を呼び出す。対応分析の道具`prince`はColabに入っていないので、なければ入れる
- 入力：なし
- 出力：「セットアップ完了」の表示


In [ ]:
import importlib.util
if importlib.util.find_spec('prince') is None:   # 無ければ入れる
    %pip install -q prince
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import prince

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
print('セットアップ完了')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は、①から③までを全部回ります。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i in (1, 2, 3)
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


---


## 1. ジャンルと年代に、関係はあるか


地図を描く前に、まず、ジャンルと年代に関係があるかを数で確かめます。MovieLensのデータから、映画をジャンル（10種類）と公開の年代（1930〜2010年代）で数えたクロス表です。1つのマスが、そのジャンル・その年代の映画の本数です。


### お手本：クロス表を読み込み、カイ二乗で確かめる

- 役割：クロス表を読み込み、ジャンルと年代に関係があるかを、カイ二乗検定で確かめる
- 入力：データの置き場所とファイル名
- 出力：クロス表`ct`、カイ二乗の値・自由度（degrees of freedom）・p値（p-value）

ジャンルと年代に関係がないなら、どの年代でも、ジャンルの割合は同じになるはずです。そのとき各マスに入るはずの本数を期待度数と呼びます。カイ二乗は、実際の本数と期待度数のずれを、1つの数に集めたものです。

$$
\chi^2 = \sum \frac{(\text{実際} - \text{期待})^2}{\text{期待}}
$$

p値は、「関係がない」としたとき、これ以上大きなずれが偶然に起きる確率です。小さいほど、「関係がない」とは考えにくくなります。`chi2_contingency`は、カイ二乗・p値・自由度・期待度数の4つを返します。


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
ct = pd.read_csv(f'{DATA_DIR}/movielens_genre_decade.csv',
                 index_col=0)
ct.columns = ct.columns.astype(str)   # 年代の名前を文字に
print(ct)
chi2, p, dof, E = chi2_contingency(ct)
print(round(chi2, 1), dof, p)


10ジャンル × 9年代、合わせて18544本。カイ二乗は548.4、自由度72、p値は10のマイナス74乗ほどと、とても小さい値です。「ジャンルと年代に関係がない」とすると、こんなに大きなずれはまず起きません。年代によって、ジャンルの構成は偏っています。

ただし、p値が小さいことは、偏りが「ある」ことを示すだけで、偏りが「どれくらい大きいか」は示しません。本数が多い表では、小さな偏りでもp値はとても小さくなります。どこが、どれくらい偏っているかは、2節の地図と、3節の「実際 ÷ 期待」の倍率で見ていきます。


### おさらい：配列の計算

```python
a = np.array([1, 2, 3])
a * 10          # すべての要素に、まとめて計算
```

numpyの配列は、`for`を書かなくても、まとめて計算できるのでした（第2回）。`np.outer(a, b)`は、aの各要素とbの各要素を総当たりでかけた表を作ります。


### TODO①：期待度数を、自分で計算する

関係がないときの期待度数は、「そのジャンルの合計 × その年代の合計 ÷ 総数」です。`ct.sum(axis=1)`（ジャンルの合計）と`ct.sum(axis=0)`（年代の合計）から、`np.outer`で期待度数の表`E2`を作り、`chi2_contingency`が返した`E`と一致するか確かめてください。


In [ ]:
# TODO: 期待度数 E2 を自分で計算し、E と一致するか確かめる
# ヒント: n = ct.values.sum()
#         E2 = np.outer(ct.sum(axis=1), ct.sum(axis=0)) / n
#         np.allclose(E, E2)
...


`True`。`chi2_contingency`の中でも、この「行の合計 × 列の合計 ÷ 総数」を計算しています。たとえば、ロマンス（1586本）の1930年代（242本）の期待度数は、1586 × 242 ÷ 18544 ≒ 20.7本。実際は48本なので、期待の2倍以上です。


---


## 2. 対応分析で、地図を描く


偏りがあることは、カイ二乗でわかりました。では、どこが偏っているのか。対応分析は、「実際 ÷ 期待」のずれの表を、PCAと同じ考え方で、少ない軸に縮める方法です（第9回）。ジャンル（行）も年代（列）も、同じ地図の上に点として置けます。同じ向きに原点から離れているジャンルと年代は、「その年代に、そのジャンルが多い」関係にあります。ただし、これは地図を読む目安です。確かな多さは、3節で「実際 ÷ 期待」の倍率を出して確かめます。


### おさらい：def

```python
def 名前(引数):
    return 戻り値
```

`def`の後ろが関数の名前と引数（入力）、`return`の後ろが戻り値（出力）でした（第3回）。地図を描く手順は、課題でも使うので、関数にしておきます。


### お手本：対応分析をかけて、地図を描く

- 役割：クロス表に対応分析をかけ、ジャンルと年代の座標を求めて、1枚の地図に描く
- 入力：クロス表`ct`
- 出力：対応分析`ca`、ジャンルの座標`rows`、年代の座標`cols`、地図

`prince.CA(n_components=2)`は、2本の軸に縮める対応分析です。`row_coordinates`が行（ジャンル）、`column_coordinates`が列（年代）の座標を返します。


In [ ]:
ca = prince.CA(n_components=2, random_state=42).fit(ct)
rows = ca.row_coordinates(ct)                # ジャンルの座標
cols = ca.column_coordinates(ct)             # 年代の座標

def draw_ca(ax, rows, cols):         # ジャンルは青、年代は赤
    ax.scatter(rows[0], rows[1], color='#0066cc', s=60)
    for name, (x, y) in rows.iterrows():
        ax.text(x, y, ' ' + name, color='#0066cc', fontsize=10)
    ax.scatter(cols[0], cols[1], color='#e63946', marker='^', s=60)
    for name, (x, y) in cols.iterrows():
        ax.text(x, y, ' ' + name, color='#e63946', fontsize=10)
    ax.axhline(0, color='gray', lw=0.5)       # 原点を通る線
    ax.axvline(0, color='gray', lw=0.5)
    ax.set_xlabel('次元1'); ax.set_ylabel('次元2')

fig, ax = plt.subplots(figsize=(8, 6))
draw_ca(ax, rows, cols)
plt.show()


### TODO②：寄与率を確かめる

この2本の軸で、全体の何割のずれを表せているか（寄与率）を確かめてください。


In [ ]:
# TODO: 次元1・次元2の寄与率を表示する
# ヒント: ca.eigenvalues_summary、または ca.percentage_of_variance_
...


次元1が67.3%、次元2が16.6%、合わせて83.9%。2本の軸で、ずれの8割以上を表せています。この地図は、かなり信頼して読めます。では、地図を読みます。


横軸（次元1）を見ると、年代は、右の端に古い年代（1930・1940年代が0.5前後）、左に新しい年代（1970・1980・2010年代）と並びます。次元1は時代の軸です。

ただし、「右が古く、左が新しい」と言えるのは両端です。1990年代（＋0.09）と2000年代（＋0.03）は原点のやや右にあり、1990年代は1960年代（＋0.06）より右です。これは、この2つの年代の本数が多いからです。確かめてみます。


In [ ]:
share = ct.sum() / ct.values.sum()     # 年代ごとの、本数の割合
print(share.round(3))


1990年代が22.4%、2000年代が31.3%で、2つ合わせて全体の54%です。原点は、全体の平均の構成を表します。本数の多い年代は、全体の平均そのものに近いので、原点の近くに来ます。だから、1990年代が1960年代より右にあっても、「1960年代より古い」とは読みません。

ジャンルでは、ロマンス（0.29）とドラマ（0.13）が右（古い年代の側）、SF（−0.21）とアクション（−0.19）が左（新しい年代の側）にいます。コメディ（−0.02）は原点の近くで、どの年代にも同じくらいある、時代を選ばない定番です。


---


## 3. 答えを出す：年代別の特集に、どのジャンルを


最初の問いに戻ります。地図の読みを、数でも確かめます。各マスの「実際の本数 ÷ 期待度数」を見れば、そのジャンルがその年代に何倍多いかがわかります。1より大きければ、その年代らしいジャンルです。


### お手本：実際 ÷ 期待の倍率

- 役割：各マスの、実際 ÷ 期待の倍率を表にする
- 入力：`ct`、期待度数`E`
- 出力：倍率の表`ratio`（4つの年代だけ表示）

表どうしの割り算も、配列と同じく、マスごとにまとめて計算されます。


In [ ]:
ratio = (ct / E).round(2)      # 1 より大きいほど、その年代に多い
print(ratio[['1930', '1940', '1980', '2010']])


1930年代のロマンスは期待の2.32倍、1940年代は、ロマンス（1.89倍）に次いでドラマが1.51倍。1980年代はホラーが1.51倍、2010年代はアクションが1.27倍、SFが1.25倍です。地図で、古い年代の近くにロマンス・ドラマ、新しい年代の近くにアクション・SFが置かれていたことと、合っています。


### おさらい：forと辞書

```python
d = {}
for i in range(3):
    d[i] = i * 10        # d は {0: 0, 1: 10, 2: 20}
```

`for`で1つずつ取り出し、辞書にキーと値をためていく書き方でした（第13回）。


### TODO③：1950年代と1970年代の上位3ジャンル

`for`で`'1950'`と`'1970'`の2つの年代を取り出し、`ratio`の倍率が高い順に3つのジャンルを、辞書`top3`に`{年代: {ジャンル: 倍率}}`の形でためて表示してください。


In [ ]:
# TODO: 1950・1970年代の、倍率の高い3ジャンルを辞書 top3 にためる
# ヒント: top3 = {} → for dec in ['1950', '1970']: の中で
#         s = ratio[dec].sort_values(ascending=False)
#         top3[dec] = s.head(3).to_dict()
...


1950年代は、ロマンス（1.65倍）・SF（1.47倍）・ドラマ（1.37倍）。1970年代は、ホラー（1.31倍）・SF（1.23倍）・犯罪もの（1.18倍）。1950年代にはまだロマンスとドラマが多く、SFが伸びはじめています。1970年代には、ロマンスが減り、ホラーとSFの時代になります。


### おさらい：f-string

```python
n = 3
print(f'{n}個')     # 3個
```

`f'…'`の中の`{}`に、値を埋めこめるのでした（第4回）。


### TODO④：特集の企画書を、一文で書く

1960年代について、`ratio['1960']`の倍率がいちばん高いジャンルといちばん低いジャンルを求め、f-stringで「1960年代の特集は ○○（□倍）を中心に。△△（□倍）は少なめでよい。」という一文を表示してください。


In [ ]:
# TODO: 1960年代の倍率が最も高い・低いジャンルで、企画書の一文を作る
# ヒント: r = ratio['1960'].sort_values()
#         いちばん高いのは r.index[-1]、倍率は r.iloc[-1]
#         いちばん低いのは r.index[0]
...


「1960年代の特集はAdventure（1.38倍）を中心に。Thriller（0.69倍）は少なめでよい。」冒険ものが、1960年代らしいジャンルです。

企画担当者としての答えは、年代ごとに次のジャンルを合わせることです。

- 古い年代の特集：ロマンスとドラマ（1930年代のロマンスは期待の2.32倍）
- 1980年代：ホラー
- 2010年代：アクションとSF
- どの年代にも合う定番：コメディ

根拠は2つです。ジャンルと年代には、はっきりした関係があります（カイ二乗548.4）。対応分析の地図は、横軸が時代の軸で、2本の軸でずれの83.9%を表せます。

まだ言えないことも、2つ残っています。

- 本数と人気は別：数えたのは作られた本数で、見られた回数ではない。特集で視聴が伸びるかは、別のデータで確かめる
- 3つ以上の区分：ジャンル・年代・国のように、区分が3つ以上になったら（発展）


---


## 目標に答えられたか


- 今日の目標は「年代別の特集に、どのジャンルを合わせればその時代らしさが出るかを示す」ことでした。3節の答えで、どの年代にどのジャンルを合わせましたか？
- 1節で、期待度数はどう計算しましたか？ カイ二乗は何を測る数でしたか？
- 2節の地図で、横軸（次元1）は何を表していましたか？ 原点の近くのジャンルは？
- 3節の倍率の表は、地図の読みと合っていましたか？


---


## 今日の要点


- クロス表の期待度数は「行の合計 × 列の合計 ÷ 総数」。カイ二乗は、実際と期待のずれを1つの数に集めたもの
- 対応分析は、ずれの表をPCAと同じ考え方で縮め、行と列を同じ地図に置く
- 原点から同じ向きに離れた行と列は、「その列に、その行が多い」関係。原点の近くは、どの列にも同じくらいある
- 地図の読みは、「実際 ÷ 期待」の倍率で、数でも確かめる
- Python：配列の計算、`def`、`for`と辞書、f-stringを組み合わせて使う


---


## 次回へ


次回は最終回、最終討論です。

これまでの分析から1つを選びます。その分析の「目標 → 解析 → 目標に立ち返る」の一周と、「まだ言えないこと」を、自分の言葉で説明できるでしょうか。


今日も、年代別の特集という目標を立て、地図と倍率で解析し、企画の答えを出しました。どの分析を選ぶかを考えるために、第14回までの歩みをふり返ります。題材は自転車・民泊・住宅・心臓病・ペンギン・顔・自動車・国・映画とめぐりましたが、どの回も「① 目標を立てる → ② 解析する → ③ 目標に立ち返る」の一周でした。

| 回 | 分析 | 新しく出てきた書き方 |
|---|---|---|
| 1 | データを見る（EDA） | 列を選ぶ、条件で絞る |
| 2 | 回帰と汎化（generalization） | numpyの配列、`for`、`if` |
| 3 | 前処理 | `def`、辞書、条件で置き換える |
| 4 | 評価と解釈 | 関数を組み合わせる、f-string |
| 5 | 正則化（regularization） | ドキュメントで引数を読む |
| 6 | 分類（classification） | 配列の比較とTrue/Falseの集計 |
| 7 | パイプライン（pipeline）としきい値 | クラスとメソッド（`fit`・`predict`） |
| 8 | 総合演習（教師ありを一周） | 関数に分けた1本のスクリプト |
| 9 | PCA | 行列の計算（`@`・`.T`・固有値分解） |
| 10 | PCAの読み解き | 関数に分かれたスクリプトを読む |
| 11 | クラスタリング（clustering） | `while` |
| 12 | 研究のコードを読む（銀河の画像） | ノートをJSONで読む、`glob`・`np.load`・画像の読み込み、`exec` |
| 13 | クラスタリングの妥当性（validity） | リスト・辞書に集めて比べる |
| 14 | 対応分析 | 総ざらい |

分析は、正解のラベルがあるとき（第1〜8回：当てる・測る・しきい値を決める）と、ないとき（第9〜14回：縮める・分ける・地図にする）に分かれました。どちらでも、物差しは判断の材料です。最後は目的に照らして答えを出し、「まだ言えないこと」を正直に書くという姿勢も、同じです。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

2節の地図は、1930年代から2010年代までの9つの年代で描きました。今度は、年代を1990年代以降の3つ（`'1990'`・`'2000'`・`'2010'`） に絞ったクロス表`ct_new`を作り、同じように対応分析をかけて、寄与率と地図を表示してください。地図は、2節の関数`draw_ca`を使えば描けます。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`ct_new = ct[['1990', '2000', '2010']]`で3列を取り出します。
あとは2節と同じく`prince.CA(n_components=2, random_state=42).fit(ct_new)` → `row_coordinates(ct_new)`・`column_coordinates(ct_new)` → `draw_ca(ax, …)`です。

</details>


### 応用②（判断）

応用①の、次元1の寄与率は何%ですか。%で小数第1位まで答えてください（例: 12.3）。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`ca_new.percentage_of_variance_[0]`を`round(…, 1)`します。

</details>


### 応用③（解釈）

2節の地図（9つの年代）と、応用①の地図（1990年代以降）を見比べて、いちばん変わった点は何ですか。近づいた・離れたジャンルと年代に触れながら、企画担当者に向けて3行程度で書いてください。


（ここに3行程度で書く）


---


## 発展（任意）


### 多重対応分析（MCA）：区分が3つ以上あるとき

今日の対応分析は、「ジャンル × 年代」という2つの区分のクロス表を地図にしました。アンケートのように、区分が3つも4つもあるときは、多重対応分析（MCA）を使います。それぞれの区分を「その区分なら1」の列（ワンホット、第7回）に直した大きな表に、対応分析をかける、と考えれば十分です。

第8回のペンギンの、種・島・性別の3つの区分で試します。


In [ ]:
penguins = pd.read_csv(f'{DATA_DIR}/penguins.csv')
X_cat = penguins[['species', 'island', 'sex']].dropna()   # 欠けた行は除く
mca = prince.MCA(n_components=2, random_state=42).fit(X_cat)
print(len(X_cat))
print(mca.eigenvalues_summary)
print(mca.column_coordinates(X_cat).round(2))


In [ ]:
cc = mca.column_coordinates(X_cat)
colors = {'species': '#0066cc', 'island': '#e63946', 'sex': '#2a9d8f'}
fig, ax = plt.subplots(figsize=(8, 6))
for name, (x, y) in cc.iterrows():
    var, cat = name.split('__')              # 例: 'species__Adelie'
    ax.scatter(x, y, color=colors[var], s=60)
    ax.text(x, y, f' {var}: {cat}', color=colors[var], fontsize=10)
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.margins(0.15); plt.show()


GentooとBiscoe島が左に、ChinstrapとDream島が右下に、AdelieとTorgersen島が上に並びます。原点から同じ向きに離れているので、この3組が強く結びついています。第8回のクロス集計（ChinstrapはDream島だけ、GentooはBiscoe島だけ）と合っています。

性別（Female・Male）は、原点のすぐ近くです。どの種・どの島にも、雄と雌が同じくらいいるからです。

試すなら、`penguins['size'] = pd.cut(penguins['body_mass_g'], 3, labels=['軽', '中', '重'])`のように体重を3つの区分にして4つ目の区分に加え、地図がどう変わるか見てみましょう。
